## Setup Student Projects in OpenAI

After creating projects for each student ([Instructions](https://github.com/Tulane-CMPS-1010-AI-Systems/course-materials/blob/main/instructor/OpenAI-Instructors.md)), the below will:

- set rate limits for selected models (tokens/requests per minute)
- set rate limits for all other models to 1 (because 0 is not allowed in the API ;( )

You'll need to first create an Admin API Token through the OpenAI web portal.

**nb:** As of 2026-01-14, the Admin API was buggy and incomplete -- rate limits could not be set for all models. Instead, I ended up manually going to each project in the web portal and restricting access to these three models:
- gpt-4o-mini
- gpt-3.5-turbo-0125
- text-embedding-3-small


In [ ]:
# @title Set allowed models and rate limits.
import getpass
import os
import sys
import time
import requests

# allow only these models
ALLOWED_MODELS = {"gpt-4o-mini", "gpt-3.5-turbo-0125", "text-embedding-3-small"}
# limit requests per minute
ALLOWED_RPM = 120
# limit tokens per minute
ALLOWED_TPM = 100_000
os.environ['DRY_RUN'] = '0' # set to 1 to test

os.environ["OPENAI_ADMIN_KEY"] = getpass.getpass("OpenAI Admin API key: ")
OPENAI_ADMIN_KEY = os.getenv("OPENAI_ADMIN_KEY")


BASE_URL = "https://api.openai.com/v1"
HEADERS = {
    "Authorization": f"Bearer {OPENAI_ADMIN_KEY}",
    "Content-Type": "application/json",
}

# “Block” other models by setting all their supported limits to 0
BLOCK_VALUE = 1

PROJECT_NAME_PREFIX = os.getenv("PROJECT_NAME_PREFIX")  # optional

# These are the writable fields per the API docs (model-dependent). :contentReference[oaicite:1]{index=1}
WRITABLE_LIMIT_FIELDS = [
    "max_requests_per_1_minute",
    "max_tokens_per_1_minute",
    "max_images_per_1_minute",
    "max_audio_megabytes_per_1_minute",
    "max_requests_per_1_day",
    "batch_1_day_max_input_tokens",
]

def get_all_projects():
    projects = []
    after = None
    while True:
        params = {"limit": 100}
        if after:
            params["after"] = after
        r = requests.get(f"{BASE_URL}/organization/projects", headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        projects.extend(data.get("data", []))
        if not data.get("has_more"):
            break
        after = data.get("last_id")
    return projects

def get_project_rate_limits(project_id: str):
    r = requests.get(
        f"{BASE_URL}/organization/projects/{project_id}/rate_limits",
        headers=HEADERS,
        params={"limit": 100},
        timeout=60,
    )
    r.raise_for_status()
    return r.json().get("data", [])


def update_rate_limit(project_id: str, rate_limit_id: str, payload: dict):
    if DRY_RUN:
        return {"dry_run": True, "payload": payload}

    r = requests.post(
        f"{BASE_URL}/organization/projects/{project_id}/rate_limits/{rate_limit_id}",
        headers=HEADERS,
        json=payload,
        timeout=60,
    )

    if r.ok:
        return r.json()

    # Try to parse structured error
    try:
        err = r.json().get("error", {})
    except Exception:
        err = {}

    # Skip models whose rate limit record isn't configurable for this org/model
    if (
        r.status_code == 400
        and err.get("code") == "rate_limit_does_not_exist_for_org_and_model"
    ):
        # Treat as non-configurable and move on
        return {"skipped": True, "reason": err.get("message")}

    # Otherwise: raise with body for debugging
    # try:
    #     print("    ERROR BODY:", r.json(), file=sys.stderr)
    # except Exception:
    #     print("    ERROR TEXT:", r.text, file=sys.stderr)
    # r.raise_for_status()


def model_is_allowed(model: str) -> bool:
    for a in ALLOWED_MODELS:
        if model.startswith(a):
            return True
    return False

def payload_for_rate_limit(rl: dict) -> dict:
    """
    Build a payload that only includes fields that are present on this rl object.
    This avoids 400s for models that don’t support certain limit types. :contentReference[oaicite:2]{index=2}
    """
    model = rl.get("model")
    present_fields = [f for f in WRITABLE_LIMIT_FIELDS if f in rl and rl[f] is not None]

    payload = {}
    if model_is_allowed(model):
        # Set RPM/TPM if those fields exist for this model.
        if "max_requests_per_1_minute" in present_fields:
            payload["max_requests_per_1_minute"] = ALLOWED_RPM
        if "max_tokens_per_1_minute" in present_fields:
            payload["max_tokens_per_1_minute"] = ALLOWED_TPM
        # For any other present fields (images/audio/day caps), leave unchanged.
    else:
        # Block non-allowed models by zeroing out *all* supported fields present on this object.
        for f in present_fields:
            payload[f] = BLOCK_VALUE

    return payload

def main():
    projects = get_all_projects()
    print(f"Found {len(projects)} projects. DRY_RUN={DRY_RUN}")

    changed_projects = 0

    for p in projects:
        if p.get("status") == "archived":
            continue

        project_id = p["id"]
        project_name = p.get("name") or p.get("title") or project_id

        if PROJECT_NAME_PREFIX and not str(project_name).startswith(PROJECT_NAME_PREFIX):
            continue
        if project_name == "Default project":
            continue
        if project_name in have: # ['josiah-chase', 'marta-nahorniuk']:
            continue

        print(project_name)
        rate_limits = get_project_rate_limits(project_id)
        if not rate_limits:
            print(f"- {project_name}: no rate limits returned (skipping)")
            continue

        print(f"\n- {project_name} ({project_id})")
        project_changed = False

        for rl in rate_limits:
            rid = rl["id"]
            model = rl.get("model") or ""
            payload = payload_for_rate_limit(rl)

            if not payload:
                continue  # nothing to change / nothing supported

            # Determine whether anything would change
            would_change = any(rl.get(k) != v for k, v in payload.items())
            if not would_change:
                continue

            action = "ALLOW" if model in ALLOWED_MODELS else "BLOCK"
            changes = ", ".join([f"{k}:{rl.get(k)}→{v}" for k, v in payload.items()])
            # print(f"  • {action:5} {model:30} {changes}")

            update_rate_limit(project_id, rid, payload)
            project_changed = True
            time.sleep(.2)

        if project_changed:
            changed_projects += 1
        else:
            print("  (no changes needed)")

    print(f"\nDone. Projects updated: {changed_projects}")

if __name__ == "__main__":
    main()

## Invite users

Script below invites users from  `roster.xlsx` that looks like this:

Student       Tulane_ID  OpenAI Project

Smith, Bob    bsmith3    bob-smith

In [3]:
import getpass
import os
os.environ["OPENAI_ADMIN_KEY"] = getpass.getpass("OpenAI Admin API key: ")
OPENAI_ADMIN_KEY = os.getenv("OPENAI_ADMIN_KEY")


OpenAI Admin API key: ··········


In [ ]:
# @title Send invites to class roster.
import os
import sys
import csv
import time
from typing import Dict, List, Tuple, Optional

import requests

try:
    import openpyxl  # type: ignore
except ImportError:
    openpyxl = None

BASE_URL = "https://api.openai.com/v1"
DRY_RUN = None # set to None to really run, otherwise "1"
XLS_FILE = "roster.xlsx"

# Column headers expected in the sheet
COL_TULANE_ID = "Tulane_ID"
COL_PROJECT = "OpenAI Project"

ORG_ROLE_FOR_INVITE = "reader"      # org-level role in invite request
PROJECT_ROLE = "member"             # project-level role in invite / project user

HEADERS = {
    "Authorization": f"Bearer {OPENAI_ADMIN_KEY}",
    "Content-Type": "application/json",
}

def die(msg: str) -> None:
    print(msg, file=sys.stderr)
    sys.exit(1)

def request_json(method: str, path: str, *, params=None, json=None) -> dict:
    url = f"{BASE_URL}{path}"
    r = requests.request(method, url, headers=HEADERS, params=params, json=json, timeout=30)
    if not r.ok:
        # print body for debugging
        try:
            body = r.json()
        except Exception:
            body = r.text
        raise requests.HTTPError(f"{r.status_code} {r.reason} for {url}\n{body}", response=r)
    return r.json()

def paginate(path: str, *, params=None, limit=100) -> List[dict]:
    out: List[dict] = []
    after = None
    while True:
        p = dict(params or {})
        p["limit"] = limit
        if after:
            p["after"] = after
        data = request_json("GET", path, params=p)
        items = data.get("data", [])
        out.extend(items)
        if not data.get("has_more"):
            break
        after = data.get("last_id")
    return out

def load_rows(filepath: str) -> List[dict]:
    if filepath.lower().endswith(".csv"):
        with open(filepath, newline="", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            return [row for row in reader]

    if filepath.lower().endswith(".xlsx"):
        if openpyxl is None:
            die("openpyxl not installed. Run: pip install openpyxl")
        wb = openpyxl.load_workbook(filepath)
        ws = wb.active
        rows = list(ws.iter_rows(values_only=True))
        if not rows:
            return []
        headers = [str(h).strip() if h is not None else "" for h in rows[0]]
        out = []
        for r in rows[1:]:
            d = {headers[i]: (r[i] if i < len(r) else None) for i in range(len(headers))}
            out.append({k: ("" if v is None else str(v).strip()) for k, v in d.items()})
        return out

    die("Unsupported file type. Use .csv or .xlsx")

def normalize(s: str) -> str:
    return " ".join(s.strip().lower().split())

def main():
    OPENAI_ADMIN_KEY = os.getenv("OPENAI_ADMIN_KEY")
    filepath = XLS_FILE

    rows = load_rows(filepath)
    if not rows:
        die("No rows found.")

    # Fetch org projects (name -> id)
    projects = paginate("/organization/projects", params={"include_archived": False})
    project_by_name: Dict[str, str] = {}
    collisions: Dict[str, List[str]] = {}
    for p in projects:
        name = p.get("name", "")
        pid = p.get("id")
        if not name or not pid:
            continue
        key = normalize(name)
        if key in project_by_name and project_by_name[key] != pid:
            collisions.setdefault(key, []).extend([project_by_name[key], pid])
        project_by_name[key] = pid

    if collisions:
        print("WARNING: Multiple projects share the same normalized name. Matching may be ambiguous:", file=sys.stderr)
        for k, ids in collisions.items():
            print(f"  - {k}: {sorted(set(ids))}", file=sys.stderr)

    # Fetch org users (email -> user_id)
    users = paginate("/organization/users")
    user_id_by_email = {normalize(u.get("email", "")): u.get("id") for u in users if u.get("email") and u.get("id")}

    # Fetch existing invites (email -> status)
    invites = paginate("/organization/invites")
    invite_by_email = {normalize(i.get("email", "")): i for i in invites if i.get("email")}

    created_invites = 0
    added_to_project = 0
    skipped = 0
    errors = 0

    for row in rows:
        tulane_id = (row.get(COL_TULANE_ID) or "").strip()
        project_name = (row.get(COL_PROJECT) or "").strip()

        if not tulane_id or not project_name:
            print(f"SKIP: missing {COL_TULANE_ID} or {COL_PROJECT} in row: {row}")
            skipped += 1
            continue

        email = f"{tulane_id}@tulane.edu"
        proj_id = project_by_name.get(normalize(project_name))
        if not proj_id:
            print(f"ERROR: project name not found in org projects: '{project_name}' (email={email})")
            errors += 1
            continue

        email_key = normalize(email)

        try:
            if email_key in user_id_by_email:
                # User already in org -> add to project directly
                user_id = user_id_by_email[email_key]
                payload = {"user_id": user_id, "role": PROJECT_ROLE}
                if DRY_RUN:
                    print(f"DRY_RUN add user to project: {email} -> {proj_id} ({project_name}) payload={payload}")
                else:
                    # Create project user: POST /organization/projects/{project_id}/users :contentReference[oaicite:3]{index=3}
                    request_json("POST", f"/organization/projects/{proj_id}/users", json=payload)
                    print(f"Added org user to project: {email} -> {project_name}")
                added_to_project += 1
            else:
                # Not in org -> invite them, granting project membership upon acceptance
                existing = invite_by_email.get(email_key)
                if existing and existing.get("status") == "pending":
                    print(f"SKIP (already invited, pending): {email} -> {project_name}")
                    skipped += 1
                    continue

                payload = {
                    "email": email,
                    "role": ORG_ROLE_FOR_INVITE,
                    "projects": [{"id": proj_id, "role": PROJECT_ROLE}],
                }
                if DRY_RUN:
                    print(f"DRY_RUN invite: {email} -> {proj_id} ({project_name}) payload={payload}")
                else:
                    # Create invite: POST /organization/invites :contentReference[oaicite:4]{index=4}
                    request_json("POST", "/organization/invites", json=payload)
                    print(f"Invited (org + project): {email} -> {project_name}")
                created_invites += 1

        except requests.HTTPError as e:
            print(f"ERROR for {email} -> {project_name}: {e}", file=sys.stderr)
            errors += 1

        time.sleep(0.1)  # be polite

    print("\nSummary:")
    print(f"  Added to project (already org user): {added_to_project}")
    print(f"  Invites created: {created_invites}")
    print(f"  Skipped: {skipped}")
    print(f"  Errors: {errors}")
    if DRY_RUN:
        print("  (DRY_RUN=1: no changes were made)")

if __name__ == "__main__":
    DRY_RUN

    main()
